# Opportunity scenarios

This notebook demonstrates a transparent, scenario-only multi-criteria ranking. The values are deterministic illustrative aggregates and are **not** observed geographic demand, a market forecast, patient targeting, or a clinical recommendation. NHANES/MEPS survey estimates, CMS Part D provider geography, and ClinicalTrials.gov study-country activity remain separate panels.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from ckd_intelligence.analysis.opportunity import rank_opportunities  # noqa: E402

In [ ]:
components = ('need', 'screening_gap', 'prescribing', 'trial_activity')
rows = []
for key, values in {
    'Illustrative scenario A': (0.80, 0.70, 0.60, 0.50),
    'Illustrative scenario B': (0.45, 0.55, 0.65, 0.75),
    'Illustrative scenario C': (0.60, 0.40, 0.80, 0.55),
}.items():
    evidence = {}
    for component, value in zip(components, values, strict=True):
        evidence[component] = {
            'value': value,
            'source_population': f'illustrative {component} aggregate',
            'grain': 'scenario aggregate',
            'evidence_type': 'public_synthetic',
            'provenance': 'notebook deterministic scenario input',
            'geography_scope': 'scenario',
            'time_period': '2024',
            'compatibility_key': 'demo',
            'coverage': 1.0,
            'limitations': ['Illustrative; not an observed geographic estimate.'],
        }
    rows.append({
        'key': key, 'geography': key, 'time_period': '2024',
        'scenario_only': True,
        'compatibility_key': 'demo',
        'components': evidence,
    })


In [ ]:
weights = {'need': 0.30, 'screening_gap': 0.30, 'prescribing': 0.20, 'trial_activity': 0.20}
sensitivity = [
    {'need': 0.40, 'screening_gap': 0.30, 'prescribing': 0.20, 'trial_activity': 0.10},
    {'need': 0.20, 'screening_gap': 0.20, 'prescribing': 0.30, 'trial_activity': 0.30},
]
result = rank_opportunities(rows, weights, sensitivity_weights=sensitivity)
[(row.rank, row.key, row.score, row.sensitivity.rank_low, row.sensitivity.rank_high)
 for row in result.rows]

## Interpretation

The ranking is a transparent decision aid for a fictional scenario. Inspect the component evidence, normalization parameters, limitations, coverage gates, and sensitivity intervals before drawing any conclusion. A rank that moves across plausible weights is weight-sensitive; it is not a reliable priority signal.